# 05 Search B: data-protection rules by jurisdiction

**Purpose.** This notebook turns our hand-made Search B research into three tables: `jurisdiction_rules.csv` (what the law says about sending personal data abroad to an AI provider, for Hong Kong, mainland China, Singapore and Taiwan), `jurisdiction_sources.csv` (every source with its type and whether we could read it) and `financial_regulator_pages.csv` (what financial regulators say about AI). Q1 uses the first as the "data-control needs" half of the question.

**Question served.** Q1 (data-control needs), Q5.

**This dataset is not downloaded.** It is our own research table, built on 16 October 2026 (financial-regulator pages on 18 October 2026) from law-firm summaries, one statute site and one unofficial translation. Rows are typed in this notebook so every value and link is visible.

**How to run.** Run all cells from top to bottom. No internet is needed.

## At a glance: links, authorisation, download, timing

| Item | Detail |
|---|---|
| Kind | Hand-built tables, typed into this notebook. Nothing is downloaded |
| Where it comes from | Fifteen sources about four places (ten read, five found but not read) and four financial-regulator pages. Every link is in Section 4 and Section 6 and in `processed/jurisdiction_sources.csv` |
| Licence and authorisation | Mixed. Law-firm articles are the firms' copyright (cite and link only). The PIPL text is an unofficial translation. HKMA allows non-commercial reproduction with attribution. APRA is Creative Commons Attribution. MAS and PCPD terms still to be recorded. No login or scraping. Details in Section 1 |
| Expected time | notebook under 2 seconds (measured 1.2 seconds). The real cost is the human reading time |
| Data as of | sources read on 2026-10-16, regulator pages on 2026-10-18. Source dates are in `jurisdiction_sources.csv` |
| To update | Re-read the sources by hand. See the update section below |

## 1. Sources, reference links and authorisation

The sources are the ones listed in Section 6 (ten read, five found but not read) and the four regulator pages in Section 7. Every row in the rules table carries the source ids it rests on.

**Authorisation and reuse.**

- Law-firm articles are copyright of the firms. We record the rule in our own words and link the article. We do not copy it.
- The PIPL text at China Law Translate is an **unofficial, crowdsourced translation** and has no date shown. Treat it as a reading aid and not as the law.
- Singapore Statutes Online is the official statute site. Singapore Statutes Online terms of use were not separately recorded in our note, so we cite and link only.
- HKMA: the HKMA owns the copyright of its pages. Its terms (paragraph 12) allow non-commercial display, download, reproduction and printing in the original format, reproduced accurately without modification, with clear attribution to the HKMA and the website. Linking and other uses need prior written consent (paragraph 13). For us: cite and link, describe the rule in our own words, do not alter or re-present the text. Terms page: https://www.hkma.gov.hk/eng/other-information/terms-and-conditions-of-use/ (read 18 Oct 2026)
- APRA: the site says "All material presented on this website is provided under a Creative Commons Attribution 4.0 International licence" (https://www.apra.gov.au/copyright). Our Search B note recorded CC BY 3.0 Australia for the CPG 230 page itself. Keep the page-level licence and cite © Australian Prudential Regulation Authority 2024.
- MAS: the terms page (https://www.mas.gov.sg/terms-of-use) did not load for our tool. A team member read it in a browser and the general terms are still to be recorded.
- PCPD (Hong Kong): two PDFs were read directly on 2026-10-18. The June 2024 Model Framework (saved as `ai_protection_framework_pcpd_hl.pdf`) and the August 2026 agentic AI supplementary guidance both state on their last page that they are licensed under Creative Commons Attribution 4.0 International with attribution to the Office of the Privacy Commissioner for Personal Data, Hong Kong. The web address of the 2024 framework is not recorded.
- No account, login or scraping is involved. Two official sites (Taiwan's law database and Singapore's PDPC site) refuse automated readers, so we did not read them and did not try to get around the block.

This is a research summary and not legal advice.

## 2. Table structure and coding scheme

### 2.1 `processed/jurisdiction_rules.csv` (4 rows, one per place)

| Column | Type | Meaning |
|---|---|---|
| `place` | text | Hong Kong, Mainland China, Singapore, Taiwan |
| `law` | text | the statute |
| `regulator` | text | the regulator |
| `transfer_rule` | text | what the law says about sending personal data abroad, in our words |
| `status` | text | in_force, not_in_operation (text exists but the commencement date has never been set), passed_not_effective (enacted, effective date still to be set) |
| `status_note` | text | dates and detail behind `status` |
| `ai_guidance` | text | AI-specific guidance found, or "Not checked" |
| `penalty_headline` | text | the main penalty, or "Not checked" |
| `source_ids` | text | ids from `jurisdiction_sources.csv`, joined with a vertical bar |
| `date_viewed` | date | 2026-10-16 |
| `caveat` | text | what limits the row |

### 2.2 `processed/jurisdiction_sources.csv` (one row per source)

| Column | Type | Meaning |
|---|---|---|
| `source_id` | text | S01, S02 and so on |
| `place` | text | place the source is about |
| `title` | text | author and title |
| `source_type` | text | primary_law, regulator_page, law_firm_summary, unofficial_translation, blog |
| `source_date` | date or empty | date printed on the source, empty if none |
| `url` | text | link |
| `read_status` | text | read (opened and text came back) or found_not_read (empty or blocked) |
| `date_viewed` | date | 2026-10-16 |
| `note` | text | what the source was used for or why it was not read |

`blog` is an addition to the three source types in the Search B brief. It marks an individual's blog, which is weaker evidence than a law-firm summary.

### 2.3 `processed/financial_regulator_pages.csv` (one row per regulator page)

| Column | Type | Meaning |
|---|---|---|
| `place`, `regulator`, `page` | text | where and which page |
| `page_date` | date or empty | date on the page |
| `what_page_shows` | text | our summary |
| `read_status` | text | opened_read or found_not_read |
| `licence_or_terms` | text | reuse terms we found |
| `url` | text | link, empty if not recorded |
| `url_recorded` | bool | false where no link is held |
| `url_origin` | text | how the link was found |
| `date_viewed` | date | 2026-10-18 (PCPD PDF is a found-not-read entry) |

## 3. Set up

In [1]:
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

T0 = time.time()
FOLDER = "05_search_b_jurisdiction_rules"
cwd = Path.cwd()
if cwd.name == FOLDER:
    HERE = cwd
elif (cwd / FOLDER).exists():
    HERE = cwd / FOLDER
elif (cwd / "Data" / FOLDER).exists():
    HERE = cwd / "Data" / FOLDER
else:
    raise FileNotFoundError(f"Open this notebook from the folder {FOLDER} (or from Data/). Current folder: {cwd}")

DATA = HERE.parent               # the Data folder
MANUAL = DATA / "raw"                # hand-downloaded or pre-copied source files
RAW = HERE / "raw"               # files this notebook downloads, kept exactly as received
OUT = HERE / "processed"         # tables this notebook writes
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW / "retrieval_log.json"
print("Notebook folder:", HERE)
print("Manual folder  :", MANUAL)

Notebook folder: d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\05_search_b_jurisdiction_rules
Manual folder  : d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\raw


## 4. Sources

In [2]:
V1 = "2026-10-16"

def src(sid, place, title, stype, sdate, url, status, note):
    return dict(source_id=sid, place=place, title=title, source_type=stype, source_date=sdate, url=url,
                read_status=status, date_viewed=V1, note=note)

SOURCES = [
    src("S01", "Hong Kong", "Bird & Bird, Hong Kong PCPD puts forward recommended model contractual clauses", "law_firm_summary", "2022",
        "https://twobirds.com/en/insights/2022/china/hong-kong-pcpd-puts-forward-recommended-model-contractual-clauses", "read",
        "Voluntary model clauses of 12 May 2022."),
    src("S02", "Hong Kong", "DLA Piper, Data protection laws of the world: Hong Kong", "law_firm_summary", "2026-03-08",
        "https://www.dlapiperdataprotection.com/index.html?c=HK", "read", "Date is the page's last-modified date."),
    src("S03", "Hong Kong", "Lewis Silkin, HR's data strategy: implementing gen AI guidelines in Hong Kong", "law_firm_summary", "2025-05-06",
        "https://www.lewissilkin.com/insights/2025/05/06/hrs-data-strategy-implementing-gen-ai-guidelines-in-hong-kong", "read",
        "PCPD checklist on employees' use of generative AI."),
    src("S04", "Hong Kong", "UPPERSETUP, PDPO Cap. 486 in 2026: the reform that never came", "blog", "2026-09-01",
        "https://uppersetup.com/en/blog/pdpo-cap-486-in-2026-the-reform-that-never-came-and", "read",
        "An individual's blog. First 100,000 characters read. Use only for the 'no bill on section 33' point and confirm with the PCPD."),
    src("S05", "Mainland China", "China Law Translate, Personal Information Protection Law (unofficial translation)", "unofficial_translation", None,
        "https://www.chinalawtranslate.com/en/Personal-Information-Protection-Law/", "read", "Crowdsourced translation, no date shown."),
    src("S06", "Mainland China", "Han Kun, March 2024 cross-border data provisions", "law_firm_summary", None,
        "https://www.hankunlaw.com/en/portal/article/index/cid/8/id/14070.html", "read", "Volume thresholds of the 22 March 2024 provisions."),
    src("S07", "Singapore", "Singapore Statutes Online, PDPA 2012 section 26", "primary_law", None,
        "https://sso.agc.gov.sg/Act/PDPA2012?ProvIds=pr26-", "read", "Page shows no version date."),
    src("S08", "Singapore", "Rajah & Tann, PDPC issues advisory guidelines on use of personal data in AI recommendation and decision systems", "law_firm_summary", "2024-03-04",
        "https://www.rajahtannasia.com/viewpoints/pdpc-issues-advisory-guidelines-on-use-of-personal-data-in-ai-recommendation-and-decision-systems/", "read",
        "Summary of the PDPC AI guidelines."),
    src("S09", "Taiwan", "Baker McKenzie, Taiwan amendment to the Personal Data Protection Act", "law_firm_summary", "2025-10-29",
        "https://www.bakermckenzie.com/en/insight/publications/alerts/2025/10/taiwan-amendment-to-personal-data-protection-act", "read",
        "Reported passage on 17 October 2025."),
    src("S10", "Taiwan", "K&L Gates, New developments in the Taiwan Personal Data Protection Act", "law_firm_summary", "2026-01-13",
        "https://www.klgates.com/New-Developments-in-the-Taiwan-Personal-Data-Protection-Act-1-13-2026", "read",
        "Promulgation 11 November 2025, effective date not yet set."),
    src("S11", "Hong Kong", "PCPD, AI Privacy Protection page", "regulator_page", None,
        "https://www.pcpd.org.hk/english/artificial_intelligence/index.html", "found_not_read", "Page text came back empty through our page reader. We did not try another way around that."),
    src("S12", "Hong Kong", "PCPD, press release on the generative AI employee checklist (31 March 2025)", "regulator_page", "2025-03-31",
        "https://www.pcpd.org.hk/english/news_events/media_statements/press_20250331.html", "found_not_read", "Page text came back empty."),
    src("S13", "Hong Kong", "PCPD, Guidance on Recommended Model Contractual Clauses (PDF)", "regulator_page", None,
        "https://www.pcpd.org.hk/english/resources_centre/publications/files/guidance_model_contractual_clauses.pdf", "found_not_read", "PDF text came back empty."),
    src("S14", "Taiwan", "Ministry of Justice, Personal Data Protection Act (English)", "primary_law", None,
        "https://www.law.moj.gov.tw/ENG/LawClass/LawAll.aspx?pcode=I0050021", "found_not_read", "Site refuses automated readers. Article 21 wording therefore not verified from the source."),
    src("S15", "Singapore", "PDPC, Advisory Guidelines on use of personal data in AI recommendation and decision systems", "regulator_page", "2024-03-01",
        "https://www.pdpc.gov.sg/guidelines-and-consultation/2024/03/advisory-guidelines-on-use-of-personal-data-in-ai-recommendation-and-decision-systems", "found_not_read",
        "Site refuses automated readers. Guideline text therefore not verified from the source."),
]
print(len(SOURCES), "sources typed")

15 sources typed


## 5. The rules table

Rule text is our summary of the sources named in `source_ids`. Where a row rests only on law-firm summaries the caveat says so.

In [3]:
RULES = [
    dict(place="Hong Kong", law="Personal Data (Privacy) Ordinance, Cap. 486", regulator="PCPD (Office of the Privacy Commissioner for Personal Data)",
         transfer_rule="Section 33 (cross-border transfer) has been in the Ordinance since 1996 but is not in operation, so there is no statutory transfer restriction. The PCPD issued voluntary Recommended Model Contractual Clauses on 12 May 2022 for transfers to another data user or to a data processor. A voluntary Greater Bay Area Standard Contract was extended to all sectors from November 2024.",
         status="not_in_operation", status_note="Section 33 never commenced. No bill or consultation on it as of August 2026 per one blog (S04).",
         ai_guidance="PCPD AI Model Personal Data Protection Framework (June 2024). PCPD checklist on employees' use of generative AI (2025). Neither is a statutory code.",
         penalty_headline="No administrative fines. Ignoring an enforcement notice: up to HK$50,000 and 2 years' prison on first conviction. Mandatory breach notification does not exist.",
         source_ids="S01|S02|S03|S04", caveat="All read sources are law-firm summaries or a blog. The PCPD pages (S11 to S13) could not be read."),
    dict(place="Mainland China", law="Personal Information Protection Law (PIPL)", regulator="Cyberspace Administration of China (CAC)",
         transfer_rule="Article 38: a business need to send data abroad must meet a condition such as a CAC security assessment, certification or a CAC standard contract. Article 39: tell individuals about the overseas recipient and get separate consent. Article 40: operators of critical information infrastructure and handlers above a CAC volume must store data in China. Article 41: no giving China-held data to foreign courts or police without approval. March 2024 Provisions: above 1,000,000 individuals (non-sensitive) or 10,000 (sensitive) needs a security assessment. 100,000 to 1,000,000 needs a standard contract or certification. Under 100,000 non-sensitive since 1 January is exempt, with notice, consent and impact assessment still required.",
         status="in_force", status_note="PIPL effective 1 November 2021. Provisions issued 22 March 2024, effective on publication.",
         ai_guidance="Not checked. The generative AI service rules are outside this table.",
         penalty_headline="Serious violations: up to RMB 50,000,000 or 5% of prior-year business income, plus suspension of operations (Article 66).",
         source_ids="S05|S06", caveat="Article text is from an unofficial translation with no date. Thresholds are from a law-firm summary."),
    dict(place="Singapore", law="Personal Data Protection Act 2012 (PDPA)", regulator="Personal Data Protection Commission (PDPC)",
         transfer_rule="Section 26: an organisation must not transfer personal data outside Singapore except as the regulations allow, and the regulations must ensure the data gets protection comparable to the Act. The PDPC may exempt an organisation on application.",
         status="in_force", status_note="Page shows no version date.",
         ai_guidance="PDPC Advisory Guidelines on use of personal data in AI recommendation and decision systems, published 1 March 2024. Third-party AI developers may count as data intermediaries.",
         penalty_headline="Not checked.",
         source_ids="S07|S08", caveat="The PDPC guideline text itself (S15) could not be read."),
    dict(place="Taiwan", law="Personal Data Protection Act (PDPA)", regulator="Moving to a new Personal Data Protection Commission",
         transfer_rule="Article 21 lets the authority restrict international transfer. The 2025 amendment moves that power from industry authorities to the new Commission. The wording of the restriction itself was not read.",
         status="passed_not_effective", status_note="Amendment promulgated 11 November 2025. The effective date is set by the Executive Yuan and was not yet set as of 13 January 2026. Baker McKenzie (29 October 2025) reported passage on 17 October 2025.",
         ai_guidance="Not checked.",
         penalty_headline="Breach-report failure: NT$20,000 to NT$200,000. Security-measure failures: NT$20,000 to NT$2,000,000, plus NT$150,000 to NT$15,000,000 for each failure to fix.",
         source_ids="S09|S10", caveat="Article 21 wording not verified (S14 blocked). Taiwan financial-sector rules are not covered. The effective date may have moved, check again before submission."),
]
for r in RULES:
    r["date_viewed"] = V1
print(len(RULES), "rules typed")

4 rules typed


## 6. Financial-sector regulator pages

These are what a bank or insurer's own regulator says about AI. Each was opened and read on **18 October 2026** after Gemini Round 3 pointed to it. They support the regulator-question situation in Q5. They are not a full list of financial rules for the four places. Australia is not one of the four places and APRA is only a cross-check on the kind of rule a bank sees.

**Links.** The Search B note did not record the web addresses of the HKMA press release, the APRA CPG 230 page and the MAS FEAT page. On 2026-10-18 a web search matched each title and the addresses below were taken from the results (`url_origin` says so). The MAS address is the FEAT principles PDF on the MAS site and the page that was read may be a web page that links to it. Open each address once and confirm it is the page that was read. The PCPD framework PDF has no web address recorded, only the local file.

In [4]:
V2 = "2026-10-18"
REGULATOR_PAGES = [
    dict(place="Hong Kong", regulator="HKMA", page="'ART' Framework press release, 15 October 2026", page_date="2026-10-15",
         what_page_shows="Announces updated supervisory guidance, a training data sharing platform and a coming discussion paper on agentic AI.",
         read_status="opened_read", licence_or_terms="Copyright of the HKMA. Non-commercial reproduction in original form with attribution allowed under paragraph 12 of https://www.hkma.gov.hk/eng/other-information/terms-and-conditions-of-use/ . Cite only, do not copy.",
         url="https://www.hkma.gov.hk/eng/news-and-media/press-releases/2026/10/20261015-3/", date_viewed=V2),
    dict(place="Australia", regulator="APRA", page="Prudential Practice Guide CPG 230 Operational Risk Management, June 2024", page_date="2024-06-01",
         what_page_shows="Guidance and not the binding standard, which is CPS 230. Notice of an incident within 72 hours.",
         read_status="opened_read", licence_or_terms="Site-wide Creative Commons Attribution 4.0 International per https://www.apra.gov.au/copyright . Our Search B note recorded CC BY 3.0 Australia for the page. Keep the page-level licence and cite the copyright holder.",
         url="https://www.apra.gov.au/practice-guides/cpg-230", date_viewed=V2),
    dict(place="Singapore", regulator="MAS", page="FEAT principles (Fairness, Ethics, Accountability, Transparency)", page_date="2019-02-07",
         what_page_shows="Published 12 November 2018 and updated 7 February 2019. Whether the principles bind is not stated on the page.",
         read_status="opened_read", licence_or_terms="MAS general terms of use at https://www.mas.gov.sg/terms-of-use not yet recorded.",
         url="https://www.mas.gov.sg/~/media/MAS/News%20and%20Publications/Monographs%20and%20Information%20Papers/FEAT%20Principles%20Final.pdf", date_viewed=V2),
    dict(place="Hong Kong", regulator="PCPD", page="AI Model Personal Data Protection Framework (PDF, June 2024, 60 pages)", page_date="2024-06-03",
         what_page_shows="Guidance for organisations procuring or using AI, including governance considerations for procuring AI solutions. Guidance and not law. Our page reader returned no text, so the saved PDF (ai_protection_framework_pcpd_hl.pdf) was read directly on 2026-10-18.",
         read_status="opened_read", licence_or_terms="Last page: licensed under Creative Commons Attribution 4.0 International. Free to share and adapt with attribution to the Office of the Privacy Commissioner for Personal Data, Hong Kong.",
         url="https://www.pcpd.org.hk/english/resources_centre/publications/files/ai_protection_framework.pdf", date_viewed=V2),
    dict(place="Hong Kong", regulator="PCPD", page="Model Framework: Protecting Personal Data Privacy in the Use of Agentic AI (PDF, August 2026, 8 pages)", page_date="2026-08-13",
         what_page_shows="Supplementary guidance to the Model Framework. AI agents are not legal persons, so the organisation stays the data user under the PDPO and its six Data Protection Principles. Lists risks (extensive access, vulnerable plugins or skills, function creep, multi-agent risks) and recommends retention limits, avoiding third-party or outdated versions, vendor contract terms on retention and security, staff training and a security checklist. Guidance and not law.",
         read_status="opened_read", licence_or_terms="Last page: licensed under Creative Commons Attribution 4.0 International with attribution to the Office of the Privacy Commissioner for Personal Data, Hong Kong.",
         url="https://www.pcpd.org.hk/english/resources_centre/publications/files/pcpd_use_of_agentic_ai.pdf", date_viewed=V2),
]
# Addresses added on 2026-10-18 by a web search that matched each page's title. The person who read the pages did not save the address.
FOUND_BY_SEARCH = {"HKMA", "APRA", "MAS"}
GIVEN_BY_TEAM = ("pcpd_use_of_agentic_ai.pdf", "ai_protection_framework.pdf")
for r in REGULATOR_PAGES:
    r["url_recorded"] = r["url"] is not None
    if r["url"] and r["url"].endswith(GIVEN_BY_TEAM):
        r["url_origin"] = "address supplied by a team member on 2026-10-18 (pasted in chat)"
    elif r["regulator"] in FOUND_BY_SEARCH:
        r["url_origin"] = "address confirmed by a team member on 2026-10-18 (opened and checked it is the right page)"
    else:
        r["url_origin"] = "none: PDF kept as a local file, address not recorded"
print(len(REGULATOR_PAGES), "regulator pages typed")

5 regulator pages typed


Taiwan (FSC) has no regulator page yet.

## 7. Build the tables and validate them

In [5]:
sources = pd.DataFrame(SOURCES)
rules = pd.DataFrame(RULES)
regs = pd.DataFrame(REGULATOR_PAGES)
for frame, cols in ((sources, ["source_date", "date_viewed"]), (rules, ["date_viewed"]), (regs, ["page_date", "date_viewed"])):
    for col in cols:
        frame[col] = frame[col].astype(str).replace({"None": None})

problems = []
if sources["source_id"].duplicated().any():
    problems.append("duplicated source_id")
if not sources["source_type"].isin({"primary_law", "regulator_page", "law_firm_summary", "unofficial_translation", "blog"}).all():
    problems.append("source_type outside the list")
if not sources["read_status"].isin({"read", "found_not_read"}).all():
    problems.append("read_status outside the list")
if not sources["url"].str.startswith("http").all():
    problems.append("a source has no valid url")
if not rules["status"].isin({"in_force", "not_in_operation", "passed_not_effective"}).all():
    problems.append("rule status outside the list")
if rules["place"].duplicated().any() or set(rules["place"]) != {"Hong Kong", "Mainland China", "Singapore", "Taiwan"}:
    problems.append("rules must hold exactly the four places once each")

# Every source id cited in a rule must exist, and each rule should cite at least one source we could read.
known = set(sources["source_id"])
readable = set(sources.loc[sources["read_status"].eq("read"), "source_id"])
for _, r in rules.iterrows():
    cited = set(r["source_ids"].split("|"))
    if not cited <= known:
        problems.append(f"{r['place']} cites unknown ids {sorted(cited - known)}")
    if not cited & readable:
        problems.append(f"{r['place']} cites no source that we could read")
    if cited - readable:
        print(f"Note: {r['place']} cites sources not read: {sorted(cited - readable)}")
# Places in the sources table must be among the four places.
if not set(sources["place"]) <= set(rules["place"]):
    problems.append("a source names a place outside the four")

missing_urls = regs.loc[~regs["url_recorded"], ["regulator", "page"]]
print("Regulator pages whose web address still has to be added by hand:", len(missing_urls))
print(missing_urls.to_string(index=False))

print()
print("Problems:", problems if problems else "none")
assert not problems, problems
sources.groupby(["place", "read_status"]).size().unstack(fill_value=0)

Regulator pages whose web address still has to be added by hand: 0
Empty DataFrame
Columns: [regulator, page]
Index: []

Problems: none


read_status,found_not_read,read
place,,
Hong Kong,3,4
Mainland China,0,2
Singapore,1,2
Taiwan,1,2


## 8. Save

In [6]:
rules.to_csv(OUT / "jurisdiction_rules.csv", index=False, encoding="utf-8")
sources.to_csv(OUT / "jurisdiction_sources.csv", index=False, encoding="utf-8")
regs.to_csv(OUT / "financial_regulator_pages.csv", index=False, encoding="utf-8")
print("saved", len(rules), "rules,", len(sources), "sources,", len(regs), "regulator pages")

saved 4 rules, 15 sources, 5 regulator pages


## How to update

There is no file to download. Laws and guidance change, so re-check before submission.

1. Open each source in Section 4 and the four regulator pages in Section 6. Priority checks:
   - Taiwan: has the Executive Yuan set the effective date of the 2025 PDPA amendment (last reported open on 2026-01-13)?
   - Hong Kong: has section 33 of the Ordinance been brought into operation, or has a bill appeared?
   - Mainland China: any change to the March 2024 cross-border thresholds?
   - Singapore: any new PDPC guidance on AI?
2. Edit the matching entry in `RULES` (Section 5), `SOURCES` (Section 4) or `REGULATOR_PAGES` (Section 6). Update `status`, `status_note`, the dates and `source_ids`.
3. Change `V1` (sources) and `V2` (regulator pages) to the days you read them.
4. Pages that could not be read by a tool (PCPD, Taiwan's law database, the PDPC site) can be opened by hand in a browser. If you read one, change its `read_status` to `read` and write what you used it for in `note`.
5. Run all cells. Section 7 stops if a status or source type is outside the allowed list or a rule cites an unknown source.

**Update check.** Open `processed/data_currency.csv`. `data_covers_to` should be your new reading day.

## 9. Checks an organisation can ask before choosing a provider

These are questions, not legal conclusions. They connect the rules above to the provider table in notebook 02.

1. **Does the prompt contain personal data?** If staff only send non-personal text, the transfer rules mostly do not apply. If they send customer or staff data, they do.
2. **Which place's law governs the data?** Mainland data faces the PIPL volume thresholds. Hong Kong has no transfer limit in force but the PCPD expects due diligence. Singapore needs comparable protection abroad. Taiwan's rule is changing.
3. **Where does the provider store and process it?** Use the storage column in notebook 02. A Singapore server is a storage fact and not a legal-jurisdiction fact.
4. **What does the contract say about training and retention?** Use the training and retention columns in notebook 02. A contract can close a gap that the law leaves open.
5. **Is there a written internal policy?** The PCPD checklist asks organisations to say which AI tools are allowed and what data may go in.

## 10. Run time and limits

**Time to extract.** Nothing is downloaded, so the run takes **under 2 seconds**. The real cost of this dataset is the human reading time.

**Limits and known risks.**

1. Most sources are law-firm summaries. Only Singapore section 26 and the China Law Translate PIPL text are close to the law itself, and the PIPL text is an unofficial translation.
2. The PCPD pages (S11 to S13) could not be read. Taiwan's law database (S14) and the PDPC site (S15) refuse automated readers. We did not try to get around either.
3. Blog claims are weaker (S04).
4. Not covered: China's generative AI service rules, Taiwan's financial-sector rules, sector rules for health, Macau and Singapore penalty detail.
5. Dates move. Taiwan's effective date was still open on 13 January 2026. Check again before submission.
6. Three regulator-page addresses were found by a later web search and still need a one-time check by eye (see Section 6).

## 11. References

The 15 sources are listed with their web addresses in `processed/jurisdiction_sources.csv` and in Section 4. The project note is `04 Searches A B C/Search B Data Protection Rules (English).md`.

## Data currency: how up to date is this data?

Every notebook writes `processed/data_currency.csv` with the day it was extracted, the dates the data itself covers, and what we know about when the source started and how often it changes. Use these values for the "data as of" label in the app.

In [7]:
def write_currency(dataset, covers_from, covers_to, snapshot_note, origin_note, frequency):
    """Record how current the data is. extracted_utc is the moment this notebook ran."""
    row = {
        "dataset": dataset,
        "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "data_covers_from": str(covers_from)[:10] if covers_from is not None else "not stated in the data",
        "data_covers_to": str(covers_to)[:10] if covers_to is not None else "not stated in the data",
        "snapshot_note": snapshot_note,
        "source_origin_note": origin_note,
        "source_update_frequency": frequency,
    }
    pd.DataFrame([row]).to_csv(OUT / "data_currency.csv", index=False, encoding="utf-8")
    for key, value in row.items():
        print(f"{key:24s} {value}")


full = sources["source_date"].dropna()
sd = pd.to_datetime(full[full.str.len() == 10], errors="coerce").dropna()
write_currency("Search B jurisdiction rules (jurisdiction_rules.csv)",
               sd.min() if len(sd) else None, "2026-10-18",
               "hand-built table, sources read on 2026-10-16 and regulator pages on 2026-10-18. Oldest source with a full date is shown (the 2022 Bird & Bird article has only a year). Taiwan effective date last reported on 2026-01-13",
               "Statutes date from 1996 (Hong Kong section 33), 2012 (Singapore PDPA) and 2021 (China PIPL). Source dates in jurisdiction_sources.csv",
               "None. Laws and guidance change, check again before submission")

dataset                  Search B jurisdiction rules (jurisdiction_rules.csv)
extracted_utc            2026-10-17T22:31:02Z
data_covers_from         2024-03-01
data_covers_to           2026-10-18
snapshot_note            hand-built table, sources read on 2026-10-16 and regulator pages on 2026-10-18. Oldest source with a full date is shown (the 2022 Bird & Bird article has only a year). Taiwan effective date last reported on 2026-01-13
source_origin_note       Statutes date from 1996 (Hong Kong section 33), 2012 (Singapore PDPA) and 2021 (China PIPL). Source dates in jurisdiction_sources.csv
source_update_frequency  None. Laws and guidance change, check again before submission


In [8]:
elapsed = time.time() - T0
print(f"Elapsed time of this run: {elapsed:.1f} seconds")
print("Files written:")
for p in sorted(OUT.glob("*")):
    print(f"  {p.relative_to(HERE)}  ({p.stat().st_size:,} bytes)")

Elapsed time of this run: 0.5 seconds
Files written:
  processed\data_currency.csv  (629 bytes)
  processed\financial_regulator_pages.csv  (3,531 bytes)
  processed\jurisdiction_rules.csv  (3,923 bytes)
  processed\jurisdiction_sources.csv  (4,193 bytes)
